In [ ]:

# notebooks/03_ongc_experiment.ipynb
# ─────────────────────────────────────────────────────────────────────────────
# ONGC Solar Turbine — Anomaly Detection with VLT Evaluation
#
# Purpose: Transfer experiment (paper Section IV-B).
#          Validates VLT+FAR framework on real industrial SCADA data.
#
# Run cells top to bottom. Each cell states exactly what it verifies.
# Expected results (based on raw signal — 80% vibration rise over 5 days):
#   IF  multi-scale: VLT > 20 hrs, FAR = 0%
#   EWMA multi-scale: VLT > 15 hrs, FAR = 0%
# ─────────────────────────────────────────────────────────────────────────────
 

In [ ]:
 
# ── CELL 1: Imports ──────────────────────────────────────────────────────────
 
import sys, os
sys.path.insert(0, r'C:\scada')
 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
 
from src.ongc_preprocessing import load_ongc_run
from src.features import (extract_rolling_features,
                           extract_multiscale_features,
                           build_feature_matrix)
from src.models import IsolationForestDetector, EWMADetector
from src.lead_time import evaluate_all_methods, plot_alarm_timeline
from src.thresholds import threshold_sweep
 
print("Imports OK")

In [ ]:

# ── CELL 2: Load Data ────────────────────────────────────────────────────────
# Verify: failure_time printed, shapes match expected (~25618 train, ~17080 test)
 
BEFORE = r'C:\scada\data\raw\ONGC\Before_Shutdown.xlsx'
AFTER  = r'C:\scada\data\raw\ONGC\After_Shutdown.xlsx'
 
run          = load_ongc_run(BEFORE, AFTER)
df_train     = run["df_train"]
df_test      = run["df_test"]
failure_time = run["failure_time"]
vib_cols     = run["vib_cols"]
 
print(f"Train: {df_train.shape}  {df_train.index[0]} → {df_train.index[-1]}")
print(f"Test:  {df_test.shape}   {df_test.index[0]} → {df_test.index[-1]}")
print(f"Failure time: {failure_time}")
 

In [ ]:

# ── CELL 3: Single-Scale Features (W=180, 30-min windows) ───────────────────
# Baseline. W=180 at 10-sec sampling = 30 minutes per window.
# include_cross_channel=True adds DE/NDE pairwise correlations.
# Verify: X_train ~283 rows, X_test ~189 rows
 
W_SINGLE = 180
 
feat_train_ss = extract_rolling_features(
    df_train, window_size=W_SINGLE, overlap=0.5,
    feature_list=["rms", "kurtosis", "crest_factor"],
    include_cross_channel=True, channel_cols=vib_cols,
)
feat_test_ss = extract_rolling_features(
    df_test, window_size=W_SINGLE, overlap=0.5,
    feature_list=["rms", "kurtosis", "crest_factor"],
    include_cross_channel=True, channel_cols=vib_cols,
)
 
X_train_ss, feat_names_ss, ts_train_ss = build_feature_matrix(feat_train_ss)
X_test_ss,  _,             ts_test_ss  = build_feature_matrix(feat_test_ss)
 
print(f"Single-scale: train={X_train_ss.shape}, test={X_test_ss.shape}, "
      f"features={len(feat_names_ss)}")

In [ ]:

# ── CELL 4: Multi-Scale Features (W=30,60,180 → 5,10,30 min) ───────────────
# C2 contribution. Same row count as single-scale (~283/189 windows).
# Feature count = 3x single-scale (one set per window size).
# Verify: X_train same row count as Cell 3, ~3x the columns
 
WINDOW_SIZES = [30, 60, 180]
 
feat_train_ms = extract_multiscale_features(
    df_train, window_sizes=WINDOW_SIZES, overlap=0.5,
    feature_list=["rms", "kurtosis", "crest_factor"],
    channel_cols=vib_cols,
)
feat_test_ms = extract_multiscale_features(
    df_test, window_sizes=WINDOW_SIZES, overlap=0.5,
    feature_list=["rms", "kurtosis", "crest_factor"],
    channel_cols=vib_cols,
)
 
X_train_ms, feat_names_ms, ts_train_ms = build_feature_matrix(feat_train_ms)
X_test_ms,  _,             ts_test_ms  = build_feature_matrix(feat_test_ms)
 
print(f"Multi-scale:  train={X_train_ms.shape}, test={X_test_ms.shape}, "
      f"features={len(feat_names_ms)}")
 

In [ ]:
 
# ── CELL 5: Evaluate Both Scales — IF and EWMA ──────────────────────────────
# Runs IF + EWMA on single-scale AND multi-scale.
# normal_period_fraction=0.20 → first 20% of test window used for FAR.
# Verify: VLT > 0 for both methods on both scales, FAR = 0%
 
detectors = [
    IsolationForestDetector(n_estimators=200, random_state=42),
    EWMADetector(lambda_=0.2, k=3.0),
]
 
print("\n=== SINGLE-SCALE ===")
summary_ss, results_ss = evaluate_all_methods(
    detectors=detectors,
    X_train=X_train_ss, X_test=X_test_ss,
    timestamps_test=ts_test_ss, failure_time=failure_time,
    normal_period_fraction=0.20, threshold_strategy="percentile",
    threshold_percentile=97.5, alarm_persistence=3,
)
print(summary_ss[["Method", "VLT (hours)", "FAR (%)", "Valid Alarm"]].to_string(index=False))
 
# Re-instantiate so models are fresh (fit() is stateful)
detectors = [
    IsolationForestDetector(n_estimators=200, random_state=42),
    EWMADetector(lambda_=0.2, k=3.0),
]
 
print("\n=== MULTI-SCALE ===")
summary_ms, results_ms = evaluate_all_methods(
    detectors=detectors,
    X_train=X_train_ms, X_test=X_test_ms,
    timestamps_test=ts_test_ms, failure_time=failure_time,
    normal_period_fraction=0.20, threshold_strategy="percentile",
    threshold_percentile=97.5, alarm_persistence=3,
)
print(summary_ms[["Method", "VLT (hours)", "FAR (%)", "Valid Alarm"]].to_string(index=False))

In [ ]:

# ── CELL 6: Combined Comparison Table ───────────────────────────────────────
# Paper Table: Method | Scale | VLT (hrs) | FAR (%)
# Saved to results/tables/ongc_comparison.csv
 
rows = []
for _, r in summary_ss.iterrows():
    rows.append({"Method": r["Method"], "Scale": "Single (W=180)",
                 "VLT (hrs)": r["VLT (hours)"], "FAR (%)": r["FAR (%)"]})
for _, r in summary_ms.iterrows():
    rows.append({"Method": r["Method"], "Scale": "Multi (W=30,60,180)",
                 "VLT (hrs)": r["VLT (hours)"], "FAR (%)": r["FAR (%)"]})
 
comparison = pd.DataFrame(rows).sort_values(["Method", "Scale"])
print("\n=== ONGC COMPARISON TABLE ===")
print(comparison.to_string(index=False))
 
os.makedirs(r'C:\scada\results\tables', exist_ok=True)
comparison.to_csv(r'C:\scada\results\tables\ongc_comparison.csv', index=False)
print("Saved → C:\\scada\\results\\tables\\ongc_comparison.csv")

In [ ]:

# ── CELL 7: Alarm Timeline Plot (best result) ────────────────────────────────
# Paper figure: anomaly score over test window, FAT marker, failure marker.
# Uses IF multi-scale result (expected to have highest VLT).
# Verify: green FAT line appears well before red failure line
 
os.makedirs(r'C:\scada\results\figures', exist_ok=True)
 
if_result = next(r for r in results_ms if "Isolation" in r["method"])
fig = plot_alarm_timeline(
    if_result,
    save_path=r'C:\scada\results\figures\ongc_if_alarm_timeline.png',
)
plt.show()
print(f"IF multi-scale: VLT={if_result['VLT_hours']:.2f} hrs | FAR={if_result['FAR_pct']:.1f}%")

In [ ]:

# ── CELL 8: Threshold Sensitivity Sweep (IF, multi-scale) ───────────────────
# Sweep percentiles 90–99.5. Shows VLT is stable — not tuned to one value.
# This is the robustness argument for the paper.
# Verify: VLT > 0 at most percentiles, FAR stays 0%
 
if_model = IsolationForestDetector(n_estimators=200, random_state=42)
if_model.fit(X_train_ms)
scores_train_if = if_model.score(X_train_ms)
scores_test_if  = if_model.score(X_test_ms)
 
n_normal = int(len(ts_test_ms) * 0.20)
normal_end_str = str(ts_test_ms[n_normal])
 
sweep_df = threshold_sweep(
    scores_train=scores_train_if,
    scores_test=scores_test_if,
    timestamps_test=ts_test_ms,
    failure_time=failure_time,
    normal_period_end=normal_end_str,
    percentile_range=[90, 92.5, 95, 97.5, 99, 99.5],
    persistence=3,
)
print("\n=== THRESHOLD SWEEP (IF, Multi-Scale) ===")
print(sweep_df[["percentile", "VLT_hours", "FAR_pct", "valid_alarm"]].to_string(index=False))

In [ ]:

 
# ── CELL 9: RMS Trend Figure (motivating figure for paper) ──────────────────
# Shows the raw DE_X vibration rise across 5 days.
# This is the "motivating figure" in the paper — why we need early detection.
 
full_before = pd.concat([df_train, df_test])
rms_trend = full_before["LPC_DE_X_Vib"].rolling(180).mean().dropna()
 
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(rms_trend.index, rms_trend.values, color="#1976D2", lw=0.8,
        label="DE_X (30-min rolling mean)")
ax.axvline(pd.Timestamp(failure_time), color="#D32F2F", lw=2,
           linestyle="--", label="Shutdown (09:46)")
ax.set_ylabel("Vibration (mm/s)", fontsize=11)
ax.set_xlabel("Time", fontsize=11)
ax.set_title("ONGC Solar Turbine — Drive End Vibration Trend", fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d-%b %H:%M"))
fig.autofmt_xdate()
plt.tight_layout()
fig.savefig(r'C:\scada\results\figures\ongc_rms_trend.png', dpi=150)
plt.show()
print("Saved → C:\\scada\\results\\figures\\ongc_rms_trend.png")
 